In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))

from leanmetric.data.load import get_path_from_env, load_data
from leanmetric.models.trainer import train_and_compare
from leanmetric.models.tune import tune_models, DEFAULT_DROP_COLS
from leanmetric.models.inspect import compute_permutation_importance

PROCESSED_DATA_PATH = get_path_from_env("PROCESSED_DATA_PATH")
MODEL_DIR_PATH = get_path_from_env("MODEL_DIR_PATH")

In [2]:
df = load_data(PROCESSED_DATA_PATH)

### Train models

In [3]:
results_df, best_artifact = train_and_compare(
    df=df,
    output_dir=MODEL_DIR_PATH / "best_model/",
    random_state=42,
)
results_df

print(f"Best model: {best_artifact['model_name']}")
print(f"Test metrics: {best_artifact['test_metrics']}")

results_df.sort_values("test_mae")

/home/enrico/miniconda3/envs/leanmetric/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.877175e+00, tolerance: 1.267e+00
  model = cd_fast.enet_coordinate_descent(


Best model: xgboost
Test metrics: {'mae': 3.351267915613511, 'rmse': 3.966960977370942, 'r2': 0.6617060981874736, 'medae': 2.825256729125975}


,model,cv_mae,cv_rmse,cv_r2,test_mae,test_rmse,test_r2
1,random_forest,3.991148,4.824609,0.690686,3.179560,3.864928,0.678885
3,mlp_sklearn,4.328190,5.403032,0.612073,3.227325,3.947229,0.665063
2,hist_gradient_boosting,4.134378,5.015784,0.665687,3.307102,3.981302,0.659256
0,xgboost,3.965828,4.908886,0.679785,3.351268,3.966961,0.661706
7,linear_regression,6.067708,31.329429,-12.043085,3.376909,3.947476,0.665021
5,elasticnet,4.397424,10.558682,-0.481476,3.421662,4.164316,0.627209
4,lasso,4.360199,10.103822,-0.356584,3.426024,4.169792,0.626228
6,ridge,4.441255,11.106116,-0.639078,3.427590,4.167045,0.626720


In [4]:
results_df, best_artifact = train_and_compare(
    df=df,
    output_dir=MODEL_DIR_PATH / "best_model_drop/",
    random_state=42,
    drop_cols=DEFAULT_DROP_COLS
)
results_df

print(f"Best model: {best_artifact['model_name']}")
print(f"Test metrics: {best_artifact['test_metrics']}")

results_df.sort_values("test_mae")

Best model: mlp_sklearn
Test metrics: {'mae': 3.897111949846529, 'rmse': 4.8456775963771515, 'r2': 0.49523719746722095, 'medae': 3.4419031654361163}


,model,cv_mae,cv_rmse,cv_r2,test_mae,test_rmse,test_r2
2,random_forest,4.000800,4.843247,0.688291,3.140370,3.808262,0.688232
7,linear_regression,6.547463,37.451397,-17.638519,3.206192,3.758781,0.696281
4,lasso,4.583019,12.356280,-1.028854,3.264349,4.032229,0.650483
5,elasticnet,4.583401,12.339316,-1.023287,3.265822,4.031684,0.650577
6,ridge,4.626698,12.884385,-1.205986,3.266474,4.029784,0.650906
1,xgboost,3.989223,4.915936,0.678865,3.270379,3.920300,0.669618
3,hist_gradient_boosting,4.119418,5.004161,0.667235,3.316543,3.937412,0.666727
0,mlp_sklearn,3.973646,4.817905,0.691545,3.897112,4.845678,0.495237


### Tune best models

In [5]:
results_df, best_artifact = tune_models(
    df=df,
    output_dir=MODEL_DIR_PATH / "best_tuned_model/",
    random_state=42,
    n_iter=30,
    drop_cols=DEFAULT_DROP_COLS
)

results_df

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Fitting 5 folds for each of 30 candidates, totalling 150 fits


,model,best_cv_mae,test_mae,test_rmse,test_r2,best_params
0,xgboost,3.795396,3.103655,3.637837,0.715511,"{'model__subsample': 0.8, 'model__reg_lambda':..."
1,random_forest,3.904991,2.985973,3.724101,0.701859,"{'model__n_estimators': 500, 'model__min_sampl..."


### Permutation importance

In [6]:
importance_df = compute_permutation_importance(
    df=df,
    model_path=MODEL_DIR_PATH / "best_tuned_model" / "best_tuned_model.joblib",
    output_dir=MODEL_DIR_PATH / "best_tuned_model",
    drop_cols=DEFAULT_DROP_COLS
)

importance_df

,feature,importance_mean,importance_std
0,navy_bodyfat,1.913040,0.148261
1,abdomen_to_height_ratio,1.605725,0.111051
2,abdomen,0.538862,0.058103
3,abdomen_to_chest_ratio,0.365138,0.049984
4,age,0.264937,0.042477
5,biceps_to_wrist_ratio,0.194535,0.032352
6,wrist_to_height_ratio,0.160567,0.027772
7,hip,0.118461,0.016866
8,knee,0.114974,0.021904
9,chest_to_height_ratio,0.114839,0.019251


### Tune only with top importance features

In [7]:
top_features = importance_df.head(15)["feature"].to_list()

droppable_features = [
    c for c in df.columns
    if c not in top_features and c != "bodyfat"
]

results_df, best_artifact = tune_models(
    df=df,
    output_dir=MODEL_DIR_PATH / "best_tuned_model_top_features/",
    random_state=42,
    n_iter=30,
    drop_cols=DEFAULT_DROP_COLS
)

results_df

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Fitting 5 folds for each of 30 candidates, totalling 150 fits


,model,best_cv_mae,test_mae,test_rmse,test_r2,best_params
0,xgboost,3.795396,3.103655,3.637837,0.715511,"{'model__subsample': 0.8, 'model__reg_lambda':..."
1,random_forest,3.904991,2.985973,3.724101,0.701859,"{'model__n_estimators': 500, 'model__min_sampl..."
